# Re-Verification (10% Sample)

Independent re-verification of a random 10% sample of decided probes.
Decisions are saved to `reverify_decisions.jsonl` (separate from the primary annotator's file).

**Instructions:** Just Run All and start clicking Keep/Reject. Same criteria as primary verification:

| Code | Criterion |
|---|---|
| **ambiguous** | Prompt has more than one defensible answer |
| **wrong-ground-truth** | Annotation is factually wrong |
| **occluded-tiny** | Target or distractor is barely visible / heavily occluded |
| **weak-distractor** | Distractor doesn't actually tempt |
| **part-color** | Color is only on a small part of the object |
| **absent-item** | Negation: doubt about whether the non-wearer actually lacks the item |
| **other** | Anything else (add reason in notes) |

In [16]:
import json
import time
from pathlib import Path
from collections import Counter

from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.schema import load_probes, Probe

In [17]:
# ── Configuration ──────────────────────────────────────────────────

DECISIONS_FILE = Path("reverify_decisions.jsonl")
IMAGES_DIR = Path("data/images")
PROBE_FILE = "reverify"

REJECT_REASONS = [
    "ambiguous",
    "wrong-ground-truth",
    "occluded-tiny",
    "weak-distractor",
    "part-color",
    "absent-item",
    "other",
]

In [18]:
# ── Load sample probes ─────────────────────────────────────────────

with open("reverify_sample.json") as f:
    sample_ids = set(json.load(f))
print(f"Re-verification sample: {len(sample_ids)} probes")

SOURCE_FILES = [
    "spatial_distractor.json", "spatial_control.json",
    "finegrained_distractor.json", "finegrained_control.json",
    "attribute_distractor.json", "attribute_control.json",
    "attribute_topup.json",
    "negation_distractor.json", "negation_control.json",
    "negation_topup.json",
]

probes = []
for fname in SOURCE_FILES:
    pf = Path("probes") / fname
    if pf.exists():
        probes.extend(p for p in load_probes(pf) if p.probe_id in sample_ids)

probes.sort(key=lambda p: (p.pair_id or "", p.probe_id))
print(f"Found {len(probes)} probes across source files")

# Filter out already-decided (resume-safe)
decided: dict[str, dict] = {}
if DECISIONS_FILE.exists():
    with DECISIONS_FILE.open() as f:
        for line in f:
            line = line.strip()
            if line:
                d = json.loads(line)
                decided[d["probe_id"]] = d

undecided = [p for p in probes if p.probe_id not in decided]
print(f"Already decided: {len(decided)}")
print(f"Remaining: {len(undecided)}")

Re-verification sample: 191 probes
Found 191 probes across source files
Already decided: 191
Remaining: 0


In [19]:
# ── Image path resolver ────────────────────────────────────────────

def get_image_path(probe: Probe) -> Path | None:
    if probe.image_source in ("coco_train2017", "lvis_v1_train"):
        return IMAGES_DIR / "coco" / f"{int(probe.image_id):012d}.jpg"
    elif probe.image_source == "visual_genome":
        vg_dir = IMAGES_DIR / "vg"
        for ext in ("jpg", "png", "jpeg"):
            p = vg_dir / f"{probe.image_id}.{ext}"
            if p.exists():
                return p
    return None

In [20]:
# ── Visualization ─────────────────────────────────────────────────

import matplotlib
matplotlib.use("agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import io

def show_probe(probe: Probe):
    img_path = get_image_path(probe)
    if img_path is None or not img_path.exists():
        print(f"Image not found for {probe.probe_id}")
        return

    img = Image.open(img_path)
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(img)

    x1, y1, x2, y2 = probe.target_box
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=3, edgecolor="lime", facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, "TARGET", color="lime", fontsize=10,
            fontweight="bold", backgroundcolor="black")

    if probe.distractor_box is not None:
        x1, y1, x2, y2 = probe.distractor_box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=3, edgecolor="red", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, "DISTRACTOR", color="red", fontsize=10,
                fontweight="bold", backgroundcolor="black")

    ax.set_title(probe.prompt, fontsize=18, fontweight="bold", pad=15)
    ax.set_xlabel(f"{probe.phenomenon}  |  {probe.probe_id}  |  pair: {probe.pair_id}",
                  fontsize=9, color="gray")
    ax.axis("off")
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    plt.close(fig)
    buf.seek(0)
    display(widgets.Image(value=buf.read(), format="png"))

In [21]:
# ── Interactive verifier ───────────────────────────────────────────

class ProbeVerifier:
    def __init__(self, probes: list[Probe], decisions_path: Path):
        self.probes = probes
        self.decisions_path = decisions_path
        self.index = 0

        self.info_label = widgets.HTML()
        self.output = widgets.Output()
        self.reason_dropdown = widgets.Dropdown(
            options=REJECT_REASONS, value="ambiguous",
            description="Reason:", layout=widgets.Layout(width="250px"))
        self.notes_input = widgets.Text(
            placeholder="Optional notes", description="Notes:",
            layout=widgets.Layout(width="400px"))
        self.keep_btn = widgets.Button(
            description="Keep", button_style="success",
            layout=widgets.Layout(width="100px"))
        self.reject_btn = widgets.Button(
            description="Reject", button_style="danger",
            layout=widgets.Layout(width="100px"))
        self.flag_btn = widgets.Button(
            description="Flag", button_style="warning",
            layout=widgets.Layout(width="100px"))
        self.skip_btn = widgets.Button(
            description="Skip", button_style="",
            layout=widgets.Layout(width="100px"))
        self.progress_label = widgets.HTML()

        self.keep_btn.on_click(lambda _: self._decide("keep"))
        self.reject_btn.on_click(lambda _: self._decide("reject"))
        self.flag_btn.on_click(lambda _: self._decide("flag"))
        self.skip_btn.on_click(lambda _: self._advance())

        buttons = widgets.HBox([self.keep_btn, self.reject_btn,
                                self.flag_btn, self.skip_btn])
        controls = widgets.VBox([buttons,
                                 widgets.HBox([self.reason_dropdown,
                                               self.notes_input]),
                                 self.progress_label])
        display(controls)
        display(self.info_label)
        display(self.output)
        self._show_current()

    def _decide(self, decision: str):
        if self.index >= len(self.probes):
            return
        probe = self.probes[self.index]
        record = {
            "probe_id": probe.probe_id,
            "decision": decision,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "source_file": PROBE_FILE,
        }
        if decision == "reject":
            record["reason"] = self.reason_dropdown.value
        if self.notes_input.value.strip():
            record["notes"] = self.notes_input.value.strip()
        with self.decisions_path.open("a") as f:
            f.write(json.dumps(record) + "\n")
        self.notes_input.value = ""
        self._advance()

    def _advance(self):
        self.index += 1
        self._show_current()

    def _show_current(self):
        if self.index >= len(self.probes):
            self.info_label.value = "<b>All probes reviewed!</b>"
            with self.output:
                clear_output(wait=True)
            self._update_progress()
            return
        probe = self.probes[self.index]
        info_html = (
            f"<b>[{self.index + 1}/{len(self.probes)}]</b> &nbsp; "
            f"probe_id=<code>{probe.probe_id}</code> &nbsp; "
            f"pair=<code>{probe.pair_id}</code>"
        )
        if probe.notes:
            info_html += f"<br><i>notes: {probe.notes}</i>"
        self.info_label.value = info_html
        with self.output:
            clear_output(wait=True)
            show_probe(probe)
        self._update_progress()

    def _update_progress(self):
        counts: dict[str, int] = Counter()
        if self.decisions_path.exists():
            with self.decisions_path.open() as f:
                for line in f:
                    line = line.strip()
                    if line:
                        d = json.loads(line)
                        counts[d["decision"]] += 1
        remaining = len(self.probes) - self.index
        self.progress_label.value = (
            f"<b>Progress:</b> "
            f"<span style='color:green'>Keep: {counts.get('keep', 0)}</span> | "
            f"<span style='color:red'>Reject: {counts.get('reject', 0)}</span> | "
            f"<span style='color:orange'>Flag: {counts.get('flag', 0)}</span> | "
            f"Remaining: {remaining}"
        )

In [22]:
# ── Start ──────────────────────────────────────────────────────────

verifier = ProbeVerifier(undecided, DECISIONS_FILE)

HTML(value='')

Output()